# NB-02 · Component Analysis
Generates: encoder comparison, HNSW latency, search pipeline optimisation, ablation figures.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from utils import (
    setup_style, save_fig, load_profiling, load_encoder_comparison,
    load_results_history, PALETTE, FIG_DIR
)

setup_style()

## 1. Encoder Comparison: BGE-M3 vs BLaIR

In [ ]:
enc = load_encoder_comparison()

# Recommendation accuracy (20k users, N=99)
rec_bge   = enc['eval_results']['BGE-M3 (current)']
rec_blair = enc['eval_results']['BLaIR (legacy)']

print('=== Recommendation Accuracy (20k users, N=99) ===')
for name, r in [('BGE-M3', rec_bge), ('BLaIR', rec_blair)]:
    print(f'  {name}: HR@5={r["hr5"]:.4f} HR@10={r["hr10"]:.4f} NDCG@10={r["ndcg10"]:.4f} MRR@10={r["mrr10"]:.4f}')

# Query retrieval per group
qg = enc['query_test']['per_group']
print('\n=== Query Retrieval Precision per Group ===')
for g, vals in qg.items():
    print(f'  {g:12s}: BGE-M3={vals["BGE-M3 (current)"]:.4f}  BLaIR={vals["BLaIR (legacy)"]:.4f}')

In [ ]:
# Figure: Query retrieval precision by query type (BGE-M3 wins on Vietnamese)
groups = list(qg.keys())
bge_vals   = [qg[g]['BGE-M3 (current)'] for g in groups]
blair_vals = [qg[g]['BLaIR (legacy)']   for g in groups]

group_labels = {
    'short':    'Short (EN)',
    'medium':   'Medium (EN)',
    'long':     'Long (EN)',
    'vi_short': 'Short (VI)',
    'vi_long':  'Long (VI)',
}

x = np.arange(len(groups))
w = 0.38
fig, ax = plt.subplots(figsize=(8, 4))

b1 = ax.bar(x - w/2, bge_vals,   w, label='BGE-M3 (current)', color='#61AFEF', edgecolor='white')
b2 = ax.bar(x + w/2, blair_vals, w, label='BLaIR (legacy)',   color='#E06C75', edgecolor='white')

for bar, v in zip(b1, bge_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{v:.2f}', ha='center', va='bottom', fontsize=8)
for bar, v in zip(b2, blair_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{v:.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([group_labels[g] for g in groups], fontsize=10)
ax.set_ylabel('Precision@10')
ax.set_ylim(0, 0.72)
ax.set_title('Search query retrieval precision by query type')
ax.legend()

# Highlight Vietnamese columns
ax.axvspan(2.5, len(groups) - 0.5, alpha=0.06, color='#98C379')
ax.text(3.5, 0.67, 'Vietnamese queries', ha='center', fontsize=9, style='italic', color='#4B8B3B')

save_fig('fig_encoder_query_comparison', fig)
plt.show()

In [ ]:
# Figure: Recommendation accuracy comparison
metrics = ['hr5', 'hr10', 'ndcg10', 'mrr10']
labels  = ['HR@5', 'HR@10', 'NDCG@10', 'MRR@10']
bge_m   = [rec_bge[m]   for m in metrics]
blair_m = [rec_blair[m] for m in metrics]

x = np.arange(len(metrics))
w = 0.38
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w/2, bge_m,   w, label='BGE-M3 (current)', color='#61AFEF', edgecolor='white')
ax.bar(x + w/2, blair_m, w, label='BLaIR (legacy)',   color='#E06C75', edgecolor='white')

for xi, (b, bl) in enumerate(zip(bge_m, blair_m)):
    ax.text(xi - w/2, b + 0.008, f'{b:.3f}', ha='center', va='bottom', fontsize=8)
    ax.text(xi + w/2, bl + 0.008, f'{bl:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Score')
ax.set_ylim(0, 0.7)
ax.set_title('Encoder comparison: recommendation accuracy (20k users, N=99)')
ax.legend()
ax.text(0.5, 0.03,
        'Note: BLaIR uses 3M-item flat index; BGE-M3 uses 1.7M-item HNSW (production).',
        transform=ax.transAxes, ha='center', fontsize=8, style='italic', color='#666')

save_fig('fig_encoder_rec_accuracy', fig)
plt.show()

## 2. HNSW vs Flat Index Latency

In [ ]:
hnsw = enc['hnsw_recall']

print('=== HNSW Recall and Speedup ===')
print(f'  Mean Recall@k : {hnsw["mean_recall_at_k"]:.4f}')
print(f'  Speedup median: {hnsw["speedup_median"]:.2f}x')
print(f'  Flat  P50={hnsw["flat_p50_ms"]:.2f}ms  P95={hnsw["flat_p95_ms"]:.2f}ms')
print(f'  HNSW  P50={hnsw["hnsw_p50_ms"]:.2f}ms  P95={hnsw["hnsw_p95_ms"]:.2f}ms')

# Encode latency
enc_lat = enc['encode_latency']
print(f'\nBGE-M3 encode latency (CUDA, n=60):')
print(f'  P50={enc_lat["p50_ms"]:.2f}ms  P95={enc_lat["p95_ms"]:.2f}ms  P99={enc_lat["p99_ms"]:.2f}ms')

In [ ]:
# Figure: HNSW vs Flat latency
fig, axes = plt.subplots(1, 2, figsize=(9, 4))

# Latency comparison
ax = axes[0]
index_types = ['Flat (exact)', 'HNSW (ANN)']
p50s = [hnsw['flat_p50_ms'], hnsw['hnsw_p50_ms']]
p95s = [hnsw['flat_p95_ms'], hnsw['hnsw_p95_ms']]
x = np.arange(len(index_types))
w = 0.38
b1 = ax.bar(x - w/2, p50s, w, label='P50', color=['#5C85D6', '#98C379'], edgecolor='white')
b2 = ax.bar(x + w/2, p95s, w, label='P95', color=['#5C85D6', '#98C379'], edgecolor='white', alpha=0.6, hatch='//')
for bar, v in zip(list(b1) + list(b2), p50s + p95s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{v:.0f}ms', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(index_types)
ax.set_ylabel('Latency (ms)')
ax.set_title('(a) Search latency: Flat vs HNSW')
solid_p = mpatches.Patch(color='#555', label='P50')
hatch_p = mpatches.Patch(facecolor='#555', alpha=0.6, hatch='//', label='P95')
ax.legend(handles=[solid_p, hatch_p])
ax.annotate(f'{hnsw["speedup_median"]:.1f}× faster\n(median)',
            xy=(1, p50s[1]), xytext=(0.7, p50s[0]*0.6),
            arrowprops=dict(arrowstyle='->', color='#555'),
            fontsize=9, ha='center')

# Recall
ax2 = axes[1]
recall = hnsw['mean_recall_at_k']
ax2.barh(['HNSW Recall@k'], [recall], color='#C678DD', edgecolor='white')
ax2.barh(['Perfect Recall'], [1.0], color='#9E9E9E', alpha=0.3, edgecolor='white')
ax2.set_xlim(0, 1.1)
ax2.set_xlabel('Recall')
ax2.set_title('(b) HNSW approximate recall')
ax2.text(recall + 0.01, 0, f'{recall:.4f}', va='center', fontsize=10)
ax2.axvline(1.0, color='#9E9E9E', linestyle='--', linewidth=0.8)

save_fig('fig_hnsw_latency', fig)
plt.show()

## 3. Search Pipeline Latency Optimisation

In [ ]:
profiling = load_profiling()

# Filter to named stages
named = [(e['run_id'], e['stage'], e['key_metrics']) for e in profiling]
for rid, stage, m in named:
    print(f'{rid:10s} {stage:30s} translate={m["translate_ms"]:7.2f}ms encode={m["encode_text_ms"]:7.2f}ms e2e={m["e2e_wall_clock_ms"]:8.2f}ms')

In [ ]:
# Figure: E2E latency across optimisation stages
# Use representative runs to show progression
stages_to_show = [
    ('run_001', 'Baseline',                  profiling[0]['key_metrics']),
    ('run_006', 'Translator\nOptimised',     profiling[5]['key_metrics']),
    ('run_008', 'Quality Gate',              profiling[7]['key_metrics']),
    ('run_011', 'QG (warm)',                 profiling[10]['key_metrics']),
    ('run_016', 'Translation\nImproved',     profiling[15]['key_metrics']),
    ('run_022', 'Cold Cache',                profiling[16]['key_metrics']),
]

labels_stage = [s[1] for s in stages_to_show]
e2e_vals     = [s[2]['e2e_wall_clock_ms'] for s in stages_to_show]
trans_vals   = [s[2]['translate_ms'] for s in stages_to_show]
enc_vals     = [s[2]['encode_text_ms'] for s in stages_to_show]
other_vals   = [max(0, e - t - en) for e, t, en in zip(e2e_vals, trans_vals, enc_vals)]

x = np.arange(len(labels_stage))
fig, ax = plt.subplots(figsize=(10, 4.5))

ax.bar(x, trans_vals,  label='Translation',       color='#E06C75', edgecolor='white')
ax.bar(x, enc_vals,    bottom=trans_vals,          label='BGE-M3 encode',    color='#61AFEF', edgecolor='white')
ax.bar(x, other_vals,  bottom=[t+e for t,e in zip(trans_vals, enc_vals)],
       label='FAISS + RRF + overhead', color='#C678DD', edgecolor='white')

for xi, e2e in zip(x, e2e_vals):
    ax.text(xi, e2e + 10, f'{e2e:.0f}ms', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(labels_stage, fontsize=9)
ax.set_ylabel('Latency (ms)')
ax.set_title('Search pipeline end-to-end latency by optimisation stage')
ax.legend(loc='upper right')

baseline_e2e = e2e_vals[0]
best_e2e     = min(e2e_vals[:-1])  # exclude cold cache
ax.annotate(f'{baseline_e2e/best_e2e:.0f}× reduction\n(warm cache)',
            xy=(0, baseline_e2e), xytext=(1, baseline_e2e * 0.85),
            arrowprops=dict(arrowstyle='->', color='#555'),
            fontsize=9, ha='center')

save_fig('fig_search_pipeline_latency', fig)
plt.show()

## 4. Ablation: Incremental Component Contribution

In [ ]:
# Ablation using 20260416_112844 run (all 5 strategies, same eval pass)
# Note: DIF-SASRec here is an earlier checkpoint (0.4836 HR@10);
# the retrained model achieves 0.7745 in the canonical run.
ablation = {
    'Content Baseline':    dict(hr10=0.4353, ndcg10=0.3030),
    'GRU-SeqDQN':          dict(hr10=0.0823, ndcg10=0.0368),
    'DIF-SASRec (early)':  dict(hr10=0.4836, ndcg10=0.3737),
    'Pipeline A (Cleora)': dict(hr10=0.7626, ndcg10=0.4867),
    'System (A∪B)':        dict(hr10=0.9736, ndcg10=0.5571),  # from final canonical
}

names  = list(ablation.keys())
hr10   = [ablation[n]['hr10'] for n in names]
ndcg10 = [ablation[n]['ndcg10'] for n in names]

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#5C85D6', '#E06C75', '#61AFEF', '#98C379', '#C678DD']

x = np.arange(len(names))
w = 0.38
b1 = ax.bar(x - w/2, hr10,   w, color=colors, edgecolor='white', label='HR@10')
b2 = ax.bar(x + w/2, ndcg10, w, color=colors, edgecolor='white', alpha=0.65, hatch='//', label='NDCG@10')

for bar, v in zip(b1, hr10):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{v:.4f}', ha='center', va='bottom', fontsize=7.5, rotation=0)
for bar, v in zip(b2, ndcg10):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{v:.4f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels(['Content\nBaseline', 'GRU-\nSeqDQN', 'DIF-SASRec\n(early)', 'Pipeline A\n(Cleora)', 'System\n(A∪B)'], fontsize=9)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.08)
ax.set_title('Ablation study: incremental component contribution (100k users, N=99)')

solid_p = mpatches.Patch(color='#555', label='HR@10')
hatch_p = mpatches.Patch(facecolor='#555', alpha=0.65, hatch='//', label='NDCG@10')
ax.legend(handles=[solid_p, hatch_p], loc='upper left')

ax.text(2, 0.03, '* Earlier checkpoint\n  (pre-full retrain)',
        fontsize=8, style='italic', color='#555')

save_fig('fig_ablation', fig)
plt.show()